# MM-Net — calibration and conformal prediction

An accuracy number says how often the model is right. It says nothing about whether
the model *knows* when it is likely to be wrong, which is the property that decides
whether a clinician can triage with it. This notebook trains the headline configuration
across the ten patient-independent folds, keeps the raw stage posteriors (before HMM
smoothing), and asks two questions:

1. **Is the model calibrated?** When it says 0.8, is it right 80% of the time?
   Measured with a reliability curve and expected calibration error.
2. **Can it abstain usefully?** Split-conformal prediction produces label *sets* with a
   distribution-free coverage guarantee. A set of size 1 is a confident call; a larger
   set is the model declining to commit. The useful question is how often it can commit.

Conformal prediction needs a calibration split that is disjoint from both training and
test. We use the fold's existing validation patients for that, so no test epoch is ever
seen before it is scored.

In [ ]:
import json
import os
import sys
import time

import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)
print("device:", C.DEV, "| subjects:", len(C.SUBS))

## 1. Train the ten folds, keeping raw posteriors

`run_10fold` returns HMM-decoded labels, which is what the paper reports but is no
longer a probability. Here we repeat the same fold loop and keep the softmax output, plus
the posteriors on each fold's validation patients for conformal calibration.

In [ ]:
t0 = time.time()
TE_P, TE_Y, CA_P, CA_Y = [], [], [], []   # test posteriors/labels, calib posteriors/labels
TE_APS, TE_APY = [], []

for fi, (tr_all, te) in enumerate(C.FOLDS):
    rng = np.random.RandomState(100 + fi)
    tr_all = list(tr_all); rng.shuffle(tr_all)
    nv = max(10, len(tr_all) // 9)
    va, tr = tr_all[:nv], tr_all[nv:]

    model = C.train_fold(tr, va, "concat", [], [], seed=42)

    for s in va:                      # conformal calibration set
        sp, _ = C.subj_infer(model, s, [], [])
        CA_P.append(sp); CA_Y.append(C.DATA[s][2])
    for s in te:                      # held-out test set
        sp, apn = C.subj_infer(model, s, [], [])
        TE_P.append(sp); TE_Y.append(C.DATA[s][2])
        TE_APS.append(apn); TE_APY.append(C.DATA[s][3])
    print("fold %d done (%.1f min elapsed)" % (fi, (time.time() - t0) / 60))

P = np.concatenate(TE_P); Y = np.concatenate(TE_Y)
Pc = np.concatenate(CA_P); Yc = np.concatenate(CA_Y)
APS = np.concatenate(TE_APS); APY = np.concatenate(TE_APY)
print("\ntest epochs %d | calibration epochs %d | total %.1f min"
      % (len(Y), len(Yc), (time.time() - t0) / 60))

## 2. Calibration of the staging head

Expected calibration error bins predictions by confidence and compares the mean
confidence in each bin with the accuracy actually achieved in it. A well-calibrated
model sits on the diagonal.

In [ ]:
conf = P.max(1)
pred = P.argmax(1)
correct = (pred == Y).astype(float)

BINS = np.linspace(0, 1, 11)
idx = np.digitize(conf, BINS) - 1
ece = 0.0
print("%-14s %8s %10s %10s %8s" % ("confidence", "n", "mean conf", "accuracy", "gap"))
print("-" * 54)
for b in range(10):
    m = idx == b
    if m.sum() < 50:
        continue
    c, a = conf[m].mean(), correct[m].mean()
    ece += (m.sum() / len(conf)) * abs(c - a)
    print("%.1f-%.1f %13d %10.3f %10.3f %8.3f"
          % (BINS[b], BINS[b + 1], m.sum(), c, a, c - a))

print("\nexpected calibration error (ECE): %.4f" % ece)
print("overall accuracy (argmax, pre-HMM): %.4f" % correct.mean())
print("mean confidence:                    %.4f" % conf.mean())

## 3. Split-conformal prediction sets

Using the calibration patients' posteriors, take the $1-\alpha$ quantile of the
non-conformity score $1 - p(\text{true class})$. On test epochs, the prediction set is
every class whose posterior exceeds that threshold. Marginal coverage is guaranteed to be
at least $1-\alpha$ regardless of whether the model is calibrated; the informative
quantity is the resulting set size.

In [ ]:
score_cal = 1.0 - Pc[np.arange(len(Yc)), Yc]

print("%-8s %10s %10s %10s %10s" % ("alpha", "target", "coverage", "mean |set|", "singletons"))
print("-" * 52)
rows = []
for alpha in (0.20, 0.15, 0.10, 0.05, 0.01):
    n = len(score_cal)
    q = np.quantile(score_cal, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")
    sets = P >= (1.0 - q)
    sets[np.arange(len(P)), P.argmax(1)] = True     # never return an empty set
    cov = sets[np.arange(len(Y)), Y].mean()
    size = sets.sum(1).mean()
    single = (sets.sum(1) == 1).mean()
    rows.append(dict(alpha=alpha, coverage=float(cov), size=float(size), singleton=float(single)))
    print("%-8.2f %10.2f %10.3f %10.2f %10.3f" % (alpha, 1 - alpha, cov, size, single))

In [ ]:
# accuracy on the epochs the model is willing to commit to, at alpha = 0.10
alpha = 0.10
n = len(score_cal)
q = np.quantile(score_cal, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")
sets = P >= (1.0 - q)
sets[np.arange(len(P)), P.argmax(1)] = True
single = sets.sum(1) == 1

print("at alpha = 0.10")
print("  epochs with a singleton set : %.1f%% of all epochs" % (100 * single.mean()))
print("  accuracy on those epochs    : %.4f" % (pred[single] == Y[single]).mean())
print("  accuracy on the remainder   : %.4f" % (pred[~single] == Y[~single]).mean())
print("  accuracy over all epochs    : %.4f" % (pred == Y).mean())

print("\nper-stage singleton rate (how often the model commits, by true stage):")
for i, c in enumerate(C.CLS):
    m = Y == i
    print("  %-3s n=%6d  singleton %.3f  accuracy %.3f"
          % (c, m.sum(), single[m].mean(), (pred[m] == Y[m]).mean()))

In [ ]:
json.dump({"ece": float(ece), "accuracy_argmax": float(correct.mean()),
           "mean_confidence": float(conf.mean()), "conformal": rows},
          open(os.path.join(OUT, "calibration_conformal.json"), "w"), indent=1)
print("wrote calibration_conformal.json")

## Reading the result

Two numbers matter for a clinical reader. The **ECE** says whether the model's stated
confidence can be taken at face value. The **singleton rate at $\alpha=0.10$** says what
fraction of the night the model will commit to a single stage while still carrying a 90%
coverage guarantee — and the accuracy gap between committed and uncommitted epochs says
whether that abstention is actually selecting the hard cases, which is the property that
would make it useful for triage rather than merely conservative.